# Lambda and map(): Transforming Order Data

In this practice you will write short anonymous functions with `lambda` and apply them to every item in a list with `map()`. You will use them to convert types, calculate totals, clean names and scale values.

## Background

Much of data preparation is applying the same small change to every value: converting a type, calculating a total, tidying text. `lambda` and `map()` do this in a single readable line, without writing a full function each time.

## Data Description

All the practices in this module use **Retail Orders.csv**: 200 orders from 2025 placed with a small online retailer based in Ireland. The data is fictional and was created for teaching.

| Column Name | Description |
|---|---|
| order_id | Unique order identifier |
| order_date | Date the order was placed (YYYY-MM-DD) |
| delivery_date | Date the order was delivered; blank if not yet delivered |
| customer_name | Customer's name, as typed into the order system |
| city | Delivery city |
| product | Product ordered |
| category | Product category: Electronics, Office or Home |
| quantity | Number of units ordered |
| unit_price | Price of one unit, in euro |
| priority | Order priority: Urgent, Normal or Minor |
| status | Order status: Delivered, Shipped or Cancelled |

## 1. Load the Dataset

We import pandas and read the CSV file into a DataFrame. Keep **Retail Orders.csv** in the same folder as this notebook.

In [1]:
import pandas as pd

df = pd.read_csv("Retail Orders.csv")
df.head()

,order_id,order_date,delivery_date,customer_name,city,product,category,quantity,unit_price,priority,status
0,ORD-1001,2025-01-11,2025-01-19,Tadhg Farrell,Limerick,Filing Cabinet,Office,5,120.0,Minor,Delivered
1,ORD-1002,2025-01-11,2025-01-15,Liam Kelly,Dublin,Whiteboard,Office,3,45.0,Normal,Delivered
2,ORD-1003,2025-01-11,2025-01-13,Sean Ryan,Cork,A4 Notebook,Office,3,4.5,Normal,Delivered
3,ORD-1004,2025-01-12,2025-01-16,Sinead Ryan,Cork,Ergonomic Chair,Office,5,189.0,Normal,Delivered
4,ORD-1005,2025-01-15,2025-01-24,Tomas Lynch,Galway,A4 Notebook,Office,1,4.5,Minor,Delivered


## 2. A def Function and Its lambda Equivalent

Both functions below calculate the price including 23% VAT. The `lambda` version fits on one line and has no name of its own until we assign it to one.

In [2]:
def with_vat(net):
    return round(net * 1.23, 2)

with_vat_lambda = lambda net: round(net * 1.23, 2)

print(with_vat(45.0))
print(with_vat_lambda(45.0))

55.35
55.35


A `lambda` can take more than one argument:

In [3]:
line_total = lambda quantity, unit_price: quantity * unit_price
line_total(3, 45.0)

135.0

## 3. map() with a Built-in Function

`map()` applies a function to every item. It returns a map object, so we wrap it in `list()` to see the results. Here `float()` converts every quantity.

In [4]:
quantities = list(df["quantity"][0:6])
print(quantities)
print(list(map(float, quantities)))

[5, 3, 3, 5, 1, 5]
[5.0, 3.0, 3.0, 5.0, 1.0, 5.0]


## 4. map() with a lambda

Add VAT to the first six unit prices in one line:

In [5]:
prices = list(df["unit_price"][0:6])
print(prices)
print(list(map(lambda p: round(p * 1.23, 2), prices)))

[120.0, 45.0, 4.5, 189.0, 4.5, 149.0]
[147.6, 55.35, 5.54, 232.47, 5.54, 183.27]


## 5. map() with Two Lists

When `map()` is given two lists, the lambda receives one item from each, in pairs. This calculates every order total at once.

In [6]:
totals = list(map(lambda q, p: q * p, quantities, prices))
totals

[600.0, 135.0, 13.5, 945.0, 4.5, 745.0]

The same approach works on whole columns, so we can add an `order_total` column to the dataset:

In [7]:
df["order_total"] = list(map(lambda q, p: q * p, df["quantity"], df["unit_price"]))
df[["order_id", "quantity", "unit_price", "order_total"]].head()

,order_id,quantity,unit_price,order_total
0,ORD-1001,5,120.0,600.0
1,ORD-1002,3,45.0,135.0
2,ORD-1003,3,4.5,13.5
3,ORD-1004,5,189.0,945.0
4,ORD-1005,1,4.5,4.5


## 6. Clean Text with map()

Some customer names were typed in inconsistent capitals, such as `jACK kENNEDY`. Converting every name to lower case is a common first cleaning step. Rows 11 to 16 include several of these names.

In [8]:
names = list(df["customer_name"][11:17])
print(names)
print(list(map(lambda n: n.lower(), names)))

['jACK kENNEDY', 'Roisin Healy', 'Liam Kelly', 'Sinead Farrell', 'ailbhe lynch', 'tadhg farrell']
['jack kennedy', 'roisin healy', 'liam kelly', 'sinead farrell', 'ailbhe lynch', 'tadhg farrell']


Lambdas can chain several string methods. Here we make short labels from the product names, in lower case and with spaces replaced by hyphens:

In [9]:
products = list(df["product"][0:5])
list(map(lambda s: s.lower().replace(" ", "-"), products))

['filing-cabinet',
 'whiteboard',
 'a4-notebook',
 'ergonomic-chair',
 'a4-notebook']

## 7. Feature Scaling with map()

Machine learning models often need values on the same scale. **Min-max scaling** maps the smallest value to 0 and the largest to 1:

scaled value = (value − minimum) ÷ (maximum − minimum)

In [10]:
min_val = min(totals)
max_val = max(totals)

scaled = list(map(lambda x: round((x - min_val) / (max_val - min_val), 3), totals))
print(totals)
print(scaled)

[600.0, 135.0, 13.5, 945.0, 4.5, 745.0]
[0.633, 0.139, 0.01, 1.0, 0.0, 0.787]


## 8. Scale a Whole Column

The same formula scales all 200 order totals.

In [11]:
low = df["order_total"].min()
high = df["order_total"].max()

df["total_scaled"] = list(map(lambda x: round((x - low) / (high - low), 3), df["order_total"]))
df[["order_id", "order_total", "total_scaled"]].head()

,order_id,order_total,total_scaled
0,ORD-1001,600.0,0.633
1,ORD-1002,135.0,0.139
2,ORD-1003,13.5,0.010
3,ORD-1004,945.0,1.000
4,ORD-1005,4.5,0.000


## 9. When to Use lambda and When to Use def

`lambda` suits short, one-off transformations like the ones above. For logic with several steps, a named `def` function is easier to read and test, and `map()` works just as well with it.

In [12]:
def priority_label(priority):
    if priority == "Urgent":
        return "Same-day dispatch"
    elif priority == "Normal":
        return "Standard dispatch"
    else:
        return "Economy dispatch"

list(map(priority_label, df["priority"][0:5]))

['Economy dispatch',
 'Standard dispatch',
 'Standard dispatch',
 'Standard dispatch',
 'Economy dispatch']

## Summary

In this practice you:

- wrote `lambda` functions with one and two arguments, and compared them with `def`
- used `map()` with a built-in function, a lambda and a named function
- used `map()` with two lists to calculate order totals and add a new column
- cleaned text by mapping string methods over names and products
- scaled order totals between 0 and 1 with min-max scaling

You have now completed the practices for Introduction to Python Programming.